# 0440 · Dependencia espacial de los residuos (I de Moran por zona×mes) · versión 2

**Para qué.** El docente preguntó cómo se maneja que el modelo supone guías independientes cuando hay vecindades. Iván respondió (P2, 25-10): calcular el **I de Moran sobre los residuos de VALIDACIÓN promediados por zona y mes**, y llevar la vecindad al modelo solo si hace falta. Este cuaderno hace eso con la base V4 y la decisión v2.

> **Versión 2 (08-11-2026).** Lee las predicciones de validación que dejaron 0412 v2 a 0415 v2 (`V4/MODELADO/pred_val_*.parquet`; 0416 v2 ya no escribe `P4b_predicciones_validacion.parquet`), con cinco bloques (abril a agosto) y los modelos de `P4b_decision_v2.json`. La ubicación de las zonas es la tabla versionada `zona_localidad_v2.csv` (0430, 03-11) y la vecindad entre UPZ es `vecindad_upz_reina.csv`, ambas en la carpeta `entrega/` del repositorio. Como ahora hay cinco meses, la regla «2 de 3 meses» pasa a **mayoría de meses (3 de 5)**, con la misma lógica. Se agrega, como complemento descriptivo, el mismo cálculo sobre los residuos de la prueba sep–oct (sección 8).

**Qué usa.**
- `V4/MODELADO/pred_val_{modelo}.parquet` (versión «internas») para el modelo elegido de cada tarea y el lineal como referencia.
- `V4/MODELADO/base_modelado.parquet` de 0411 v2, solo para unir guía (seudónimo) con `zona_urbana`.
- `entrega/zona_localidad_v2.csv` y `entrega/vecindad_upz_reina.csv` (junto al cuaderno, como `MODULOS/`), o los mismos archivos subidos a `V4/ESPACIAL/`.
- Para la sección 8: `V4/MODELOS/prueba_{tarea}.parquet` de 0416_PruebaUnica_Reserva.

**Matrices de vecindad (fijadas antes de ver resultados).**
- **W1, zonas, k vecinos más cercanos** (k = 5, estandarizada por filas) sobre el punto del barrio. Es la principal.
- **W2, zonas, k = 10.** Sensibilidad.
- **W3, UPZ, contigüidad tipo reina** con los residuos promediados por UPZ y mes. Sensibilidad con unidades oficiales.

**Regla de lectura, fijada antes de correr.**
- Se calcula I con prueba de permutación (999 permutaciones) en cada mes, tarea y modelo. Solo entran zonas con al menos `MIN_GUIAS` guías en ese mes.
- **Hay dependencia que tratar** si, con W1, el modelo elegido tiene I > 0 con p < 0,05 en la mayoría de los meses de validación (3 de 5). En ese caso se propone agregar al modelo el rezago espacial de la historia (W·X, sin futuro) y reportar errores agrupados por zona.
- **Si no**, se reporta que los residuos no muestran dependencia espacial detectable a esta escala, y la independencia condicional queda como supuesto verificado. CAR o BYM quedan como trabajo futuro, como sugirió Iván.

**Escribe** solo tablas agregadas `P8_*` en `V4/ESPACIAL/`. No muestra guías.

In [0]:
%pip install "duckdb<2" scikit-learn

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 98.1 MB/s  0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CONFIGURACIÓN
import os, json, numpy as np, pandas as pd
from sklearn.neighbors import NearestNeighbors
PATH_DATABRICKS  = "/Volumes/proyectogrado/source/source/"
CARPETA_V4       = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_V4}MODELADO/"
CARPETA_MODELOS  = f"{CARPETA_V4}MODELOS/"                                    # 0416_PruebaUnica_Reserva (sección 8)
RUTA_DECISION    = f"{CARPETA_V4}P4b_decision_v2.json"
RUTA_BASE        = f"{CARPETA_MODELADO}base_modelado.parquet"                  # 0411 v2
CARPETA_ESPACIAL = f"{CARPETA_V4}ESPACIAL/"
os.makedirs(CARPETA_ESPACIAL, exist_ok=True)

def ubicar(nombre):
    """Busca el archivo en V4/ESPACIAL/ y, si no está, en entrega/ junto al cuaderno."""
    for r in [f"{CARPETA_ESPACIAL}{nombre}", os.path.join("entrega", nombre)]:
        if os.path.exists(r): return r
    raise FileNotFoundError(f"Falta {nombre}: subir la carpeta entrega/ junto al cuaderno o copiarlo a {CARPETA_ESPACIAL}")
RUTA_ZONAS   = ubicar("zona_localidad_v2.csv")       # 0430, tabla versionada del 03-11
RUTA_VEC_UPZ = ubicar("vecindad_upz_reina.csv")      # 0430, contigüidad tipo reina entre UPZ

MIN_GUIAS     = 30        # mínimo de guías de una zona en el mes para entrar al cálculo
K_PRINCIPAL   = 5
K_SENSIBLE    = 10
PERMUTACIONES = 999
SEMILLA       = 42
ALFA          = 0.05
try:
    display
except NameError:
    display = print

In [0]:
# 1. Residuos de validación por guía, con su zona
dec = json.load(open(RUTA_DECISION))
VERSION = dec["version"]
MODELOS = {t: list(dict.fromkeys([dec[f"modelo_{t}"], "Lineal"])) for t in ["clf", "reg"]}
print("Modelos evaluados:", MODELOS, "· versión", VERSION)
COLS = ["version", "tarea", "bloque", "guia", "y", "modelo", "pred"]
partes = []
for nombre in sorted({m for ms in MODELOS.values() for m in ms}):
    p = pd.read_parquet(f"{CARPETA_MODELADO}pred_val_{nombre}.parquet", columns=COLS, filters=[("version", "==", VERSION)])
    partes.append(p[[m in MODELOS[t] for t, m in zip(p.tarea, p.modelo)]])
pred = pd.concat(partes, ignore_index=True); del partes
pred["y"], pred["pred"] = pred.y.astype(float), pred.pred.astype(float)
base = pd.read_parquet(RUTA_BASE, columns=["guia", "zona_urbana"]).drop_duplicates("guia")
res = pred.merge(base, on="guia", how="left"); del pred
assert res.zona_urbana.notna().mean() > 0.95, "Muchas predicciones sin zona: revisar que 0411 y 0412–0415 sean de la misma corrida"
res["residuo"] = res.y - res.pred                      # reg: horas reales − predichas; clf: 1/0 − probabilidad
res["mes"] = pd.to_datetime(res.bloque).dt.strftime("%Y-%m")
print(res.groupby(["tarea", "modelo", "mes"]).size().unstack())

Modelos evaluados: {'clf': ['RandomForest', 'Lineal'], 'reg': ['LightGBM', 'Lineal']} · versión internas
mes                 2026-04  2026-05  2026-06  2026-07  2026-08
tarea modelo                                                   
clf   Lineal         193214   204833   185344   246075   136575
      RandomForest   193214   204833   185344   246075   136575
reg   LightGBM       191924   202331   183841   244689   135690
      Lineal         191924   202331   183841   244689   135690


In [0]:
# 2. Promedio por zona×mes y ubicación de la zona (0430)
zonas = pd.read_csv(RUTA_ZONAS)[["zona_urbana", "lon", "lat", "upz", "estado"]]
agg = (res.groupby(["tarea", "modelo", "mes", "zona_urbana"])
          .agg(guias=("residuo", "size"), residuo_medio=("residuo", "mean"), y_medio=("y", "mean")).reset_index()
          .merge(zonas, on="zona_urbana", how="left"))
cob = (agg.assign(ubicada=agg.lon.notna(), suficiente=agg.guias >= MIN_GUIAS)
          .groupby(["tarea", "modelo", "mes", "ubicada", "suficiente"]).agg(zonas=("guias", "size"), guias=("guias", "sum")).reset_index())
cob.to_csv(f"{CARPETA_ESPACIAL}P8_cobertura_moran.csv", index=False)
display(cob[(cob.tarea == "reg") & (cob.modelo == dec["modelo_reg"])])

tarea,modelo,mes,ubicada,suficiente,zonas,guias
reg,LightGBM,2026-04,false,false,221,1099
reg,LightGBM,2026-04,false,true,183,94280
reg,LightGBM,2026-04,true,false,42,260
reg,LightGBM,2026-04,true,true,142,96995
reg,LightGBM,2026-05,false,false,249,1309
reg,LightGBM,2026-05,false,true,187,96906
reg,LightGBM,2026-05,true,false,47,409
reg,LightGBM,2026-05,true,true,141,104457
reg,LightGBM,2026-06,false,false,231,1145
reg,LightGBM,2026-06,false,true,172,86795


In [0]:
# 3. I de Moran con prueba de permutación
def w_knn(xy_m, k):
    '''Matriz kNN estandarizada por filas (densa; cientos de zonas). Zonas con el mismo punto quedan vecinas a 0 m.'''
    n = len(xy_m); k = min(k, n - 1)
    _, idx = NearestNeighbors(n_neighbors=k + 1).fit(xy_m).kneighbors(xy_m)
    W = np.zeros((n, n))
    for i, fila in enumerate(idx):
        vecinos = [j for j in fila if j != i][:k]
        W[i, vecinos] = 1.0 / k
    return W

def w_contiguidad(unidades, pares):
    '''Matriz de contigüidad estandarizada por filas a partir de pares (a, b). Unidades sin vecinos quedan en cero.'''
    pos = {u: i for i, u in enumerate(unidades)}
    W = np.zeros((len(unidades), len(unidades)))
    for a, b in pares:
        if a in pos and b in pos:
            W[pos[a], pos[b]] = W[pos[b], pos[a]] = 1.0
    s = W.sum(1, keepdims=True)
    return np.divide(W, s, out=np.zeros_like(W), where=s > 0)

def moran(x, W, permutaciones=PERMUTACIONES, semilla=SEMILLA):
    z = np.asarray(x, float) - np.mean(x)
    S0 = W.sum()
    I = len(z) / S0 * (z @ W @ z) / (z @ z)
    rng = np.random.default_rng(semilla)
    sim = np.array([(lambda p: len(p) / S0 * (p @ W @ p) / (p @ p))(rng.permutation(z)) for _ in range(permutaciones)])
    p = (1 + np.sum(sim >= I)) / (permutaciones + 1)            # una cola: autocorrelación positiva
    return {"I": I, "E_I": -1 / (len(z) - 1), "p_permutacion": p, "z_sim": (I - sim.mean()) / sim.std()}

def a_metros(lon, lat):
    '''Proyección equirrectangular local (Bogotá): suficiente para vecinos más cercanos.'''
    return np.c_[np.radians(lon) * 6371000 * np.cos(np.radians(4.65)), np.radians(lat) * 6371000]

In [0]:
# 4. Moran por mes, tarea y modelo: W1 (k=5) y W2 (k=10) entre zonas
filas = []
for (tarea, modelo, mes), g in agg.groupby(["tarea", "modelo", "mes"]):
    g = g[(g.guias >= MIN_GUIAS) & g.lon.notna()].reset_index(drop=True)
    if len(g) < 20: continue
    xy = a_metros(g.lon.values, g.lat.values)
    for nombre_w, k in [("W1 zonas kNN k=5", K_PRINCIPAL), ("W2 zonas kNN k=10", K_SENSIBLE)]:
        filas.append({"tarea": tarea, "modelo": modelo, "mes": mes, "matriz": nombre_w, "unidades": len(g),
                      "guias": int(g.guias.sum()), **moran(g.residuo_medio.values, w_knn(xy, k))})
moran_zonas = pd.DataFrame(filas)

In [0]:
# 5. Sensibilidad W3: residuos por UPZ×mes con contigüidad tipo reina
filas = []
if os.path.exists(RUTA_VEC_UPZ):
    pares = pd.read_csv(RUTA_VEC_UPZ).itertuples(index=False, name=None)
    pares = list(pares)
    res_u = res.merge(zonas[["zona_urbana", "upz"]], on="zona_urbana", how="left").dropna(subset=["upz"])
    for (tarea, modelo, mes), g in res_u.groupby(["tarea", "modelo", "mes"]):
        u = g.groupby("upz").agg(guias=("residuo", "size"), residuo_medio=("residuo", "mean")).reset_index()
        u = u[u.guias >= MIN_GUIAS].reset_index(drop=True)
        if len(u) < 20: continue
        W = w_contiguidad(list(u.upz), pares)
        con_vecinos = W.sum(1) > 0
        u, W = u[con_vecinos].reset_index(drop=True), w_contiguidad(list(u.upz[con_vecinos]), pares)
        filas.append({"tarea": tarea, "modelo": modelo, "mes": mes, "matriz": "W3 UPZ reina", "unidades": len(u),
                      "guias": int(u.guias.sum()), **moran(u.residuo_medio.values, W)})
else:
    print("Sin P7_vecindad_upz.csv: se omite W3")
moran_tabla = pd.concat([moran_zonas, pd.DataFrame(filas)], ignore_index=True).round(4)
moran_tabla.to_csv(f"{CARPETA_ESPACIAL}P8_moran_residuos_validacion.csv", index=False)
display(moran_tabla)

tarea,modelo,mes,matriz,unidades,guias,I,E_I,p_permutacion,z_sim
clf,Lineal,2026-04,W1 zonas kNN k=5,142,97516,-0.0138,-0.0071,0.556,-0.1966
clf,Lineal,2026-04,W2 zonas kNN k=10,142,97516,-0.0068,-0.0071,0.47,-0.0367
clf,Lineal,2026-05,W1 zonas kNN k=5,141,105445,0.0336,-0.0071,0.206,0.8632
clf,Lineal,2026-05,W2 zonas kNN k=10,141,105445,0.0036,-0.0071,0.331,0.3443
clf,Lineal,2026-06,W1 zonas kNN k=5,139,96631,-0.0028,-0.0072,0.431,0.0991
clf,Lineal,2026-06,W2 zonas kNN k=10,139,96631,6.0E-4,-0.0072,0.361,0.2459
clf,Lineal,2026-07,W1 zonas kNN k=5,142,143060,-0.0482,-0.0071,0.841,-0.9465
clf,Lineal,2026-07,W2 zonas kNN k=10,142,143060,-0.023,-0.0071,0.695,-0.553
clf,Lineal,2026-08,W1 zonas kNN k=5,134,78808,-0.0193,-0.0075,0.551,-0.2626
clf,Lineal,2026-08,W2 zonas kNN k=10,134,78808,-0.0295,-0.0075,0.724,-0.6846


In [0]:
# 6. Referencia: Moran del OBJETIVO (no del residuo) por zona×mes con W1
#    Si el objetivo tiene dependencia y el residuo no, el modelo ya la absorbe (zona×franja, historia).
filas = []
for (tarea, mes), g in agg[agg.modelo == "Lineal"].groupby(["tarea", "mes"]):
    g = g[(g.guias >= MIN_GUIAS) & g.lon.notna()].reset_index(drop=True)
    if len(g) < 20: continue
    filas.append({"tarea": tarea, "mes": mes, "matriz": "W1 zonas kNN k=5", "unidades": len(g),
                  **moran(g.y_medio.values, w_knn(a_metros(g.lon.values, g.lat.values), K_PRINCIPAL))})
moran_y = pd.DataFrame(filas).round(4)
moran_y.to_csv(f"{CARPETA_ESPACIAL}P8_moran_objetivo.csv", index=False)
display(moran_y)

tarea,mes,matriz,unidades,I,E_I,p_permutacion,z_sim
clf,2026-04,W1 zonas kNN k=5,142,0.0052,-0.0071,0.387,0.2001
clf,2026-05,W1 zonas kNN k=5,141,0.0317,-0.0071,0.203,0.8448
clf,2026-06,W1 zonas kNN k=5,139,0.0225,-0.0072,0.248,0.6183
clf,2026-07,W1 zonas kNN k=5,142,-0.0508,-0.0071,0.863,-1.015
clf,2026-08,W1 zonas kNN k=5,134,-0.0137,-0.0075,0.496,-0.1445
reg,2026-04,W1 zonas kNN k=5,142,-0.0322,-0.0071,0.697,-0.575
reg,2026-05,W1 zonas kNN k=5,141,0.0253,-0.0071,0.254,0.6565
reg,2026-06,W1 zonas kNN k=5,139,0.0463,-0.0072,0.109,1.237
reg,2026-07,W1 zonas kNN k=5,142,-0.1071,-0.0071,0.991,-2.1207
reg,2026-08,W1 zonas kNN k=5,134,-0.0357,-0.0075,0.703,-0.5924


In [0]:
# 7. Lectura con la regla fijada (mayoría de los meses de validación)
for tarea in ["reg", "clf"]:
    elegido = dec[f"modelo_{tarea}"]
    t = moran_tabla[(moran_tabla.tarea == tarea) & (moran_tabla.modelo == elegido) & (moran_tabla.matriz == "W1 zonas kNN k=5")]
    n_meses = t.mes.nunique(); meses_sig = int(((t.I > 0) & (t.p_permutacion < ALFA)).sum())
    veredicto = ("HAY dependencia espacial en los residuos: proponer rezago W·X de la historia y errores agrupados por zona"
                 if meses_sig > n_meses / 2 else "Sin dependencia espacial detectable en los residuos: se sostiene la independencia condicional")
    print(f"{tarea} · {elegido}: I por mes = {list(t.I.round(3))}, p = {list(t.p_permutacion.round(3))} → {meses_sig}/{n_meses} meses significativos")
    print("   ", veredicto)

reg · LightGBM: I por mes = [-0.03, 0.004, 0.055, -0.094, -0.034], p = [0.698, 0.396, 0.078, 0.984, 0.686] → 0/5 meses significativos
    Sin dependencia espacial detectable en los residuos: se sostiene la independencia condicional
clf · RandomForest: I por mes = [0.0, 0.029, 0.003, -0.025, -0.016], p = [0.418, 0.213, 0.386, 0.627, 0.546] → 0/5 meses significativos
    Sin dependencia espacial detectable en los residuos: se sostiene la independencia condicional


## 8. Complemento: residuos de la prueba sep–oct

El mismo cálculo (W1 y W2) sobre los residuos de la prueba única, solo para el modelo de la decisión (es el único con predicciones guardadas en `V4/MODELOS/`). Es **descriptivo**: la regla de lectura se aplica a validación; aquí solo se verifica si el patrón se mantiene fuera de la muestra.

In [0]:
filas = []
for tarea in ["clf", "reg"]:
    ruta = f"{CARPETA_MODELOS}prueba_{tarea}.parquet"
    if not os.path.exists(ruta): print("Sin", ruta); continue
    p = pd.read_parquet(ruta, columns=["y", "pred", "dia", "zona_urbana"])
    p["residuo"] = p.y.astype(float) - p.pred.astype(float); p["mes"] = pd.to_datetime(p.dia).dt.strftime("%Y-%m")
    a = (p.groupby(["mes", "zona_urbana"]).agg(guias=("residuo", "size"), residuo_medio=("residuo", "mean")).reset_index()
          .merge(zonas, on="zona_urbana", how="left"))
    for mes, g in a.groupby("mes"):
        g = g[(g.guias >= MIN_GUIAS) & g.lon.notna()].reset_index(drop=True)
        if len(g) < 20: continue
        xy = a_metros(g.lon.values, g.lat.values)
        for nombre_w, k in [("W1 zonas kNN k=5", K_PRINCIPAL), ("W2 zonas kNN k=10", K_SENSIBLE)]:
            filas.append({"tarea": tarea, "modelo": dec[f"modelo_{tarea}"], "mes": mes, "matriz": nombre_w, "unidades": len(g),
                          "guias": int(g.guias.sum()), "pct_guias_mes": round(100 * g.guias.sum() / (p.mes == mes).sum(), 1),
                          **moran(g.residuo_medio.values, w_knn(xy, k))})
moran_prueba = pd.DataFrame(filas).round(4)
moran_prueba.to_csv(f"{CARPETA_ESPACIAL}P8_moran_residuos_prueba.csv", index=False)
display(moran_prueba)

tarea,modelo,mes,matriz,unidades,guias,pct_guias_mes,I,E_I,p_permutacion,z_sim
clf,RandomForest,2026-09,W1 zonas kNN k=5,138,114055,56.4,-0.043,-0.0073,0.787,-0.7829
clf,RandomForest,2026-09,W2 zonas kNN k=10,138,114055,56.4,-0.0422,-0.0073,0.846,-1.0442
clf,RandomForest,2026-10,W1 zonas kNN k=5,131,109698,53.0,0.0309,-0.0077,0.235,0.7208
clf,RandomForest,2026-10,W2 zonas kNN k=10,131,109698,53.0,0.0318,-0.0077,0.124,1.174
reg,LightGBM,2026-09,W1 zonas kNN k=5,138,113734,56.5,-0.0379,-0.0073,0.72,-0.6177
reg,LightGBM,2026-09,W2 zonas kNN k=10,138,113734,56.5,-0.0346,-0.0073,0.783,-0.7886
reg,LightGBM,2026-10,W1 zonas kNN k=5,131,108816,53.1,0.0112,-0.0077,0.332,0.3448
reg,LightGBM,2026-10,W2 zonas kNN k=10,131,108816,53.1,0.0323,-0.0077,0.134,1.1683


## Qué entregar

- `P8_moran_residuos_validacion.csv`: I, valor esperado, p de permutación y z por mes (abril a agosto), tarea, modelo y matriz. Es la tabla principal.
- `P8_moran_objetivo.csv`: el mismo cálculo sobre el objetivo, para mostrar cuánto de la dependencia absorbe el modelo.
- `P8_cobertura_moran.csv`: cuántas zonas y guías entran (ubicadas y con al menos `MIN_GUIAS`).
- `P8_moran_residuos_prueba.csv`: complemento descriptivo con la prueba sep–oct.

**Limitación que se declara.** Solo entran las zonas que 0430 pudo ubicar (61 % de las guías en la tabla v2) y, de ellas, las que tienen al menos `MIN_GUIAS` guías en el mes. Las zonas no geográficas (aliados, oficinas) no tienen vecindad y quedan fuera por diseño.